In [ ]:
import re
import json
from datetime import datetime, timedelta
from dataclasses import dataclass, asdict
from typing import Optional, Tuple
import dateparser
import pytz

In [10]:
KYIV_TZ = pytz.timezone("Europe/Kiev")

UK_MONTH_MAP = {
    "січня": 1, "січень": 1,
    "лютого": 2, "лютий": 2,
    "березня": 3, "березень": 3,
    "квітня": 4, "квітень": 4,
    "травня": 5, "травень": 5,
    "червня": 6, "червень": 6,
    "липня": 7, "липень": 7,
    "серпня": 8, "серпень": 8,
    "вересня": 9, "вересень": 9,
    "жовтня": 10, "жовтень": 10,
    "листопада": 11, "листопад": 11,
    "грудня": 12, "грудень": 12,
}
UK_MONTHS = "(?:" + "|".join(UK_MONTH_MAP.keys()) + ")"

In [11]:
PLANNED_SIGNALS = [
    r"планов", r"планується", r"заплановано", r"оголошу", r"повідомля",
    r"буде\s+(?:проведено|відключено|обмежено|закрито)",
    r"на\s+період", r"тимчасово",
    r"(?:захід|подія|ремонт|перекриття)\s+(?:відбудеться|пройде|буде)",
]
PLANNED_RE = re.compile("|".join(PLANNED_SIGNALS), re.IGNORECASE)
RELATIVE_HINTS = ["сьогодні", "вчора", "завтра", "зранку", "ввечері", "до вечора", "до кінця дня"]

In [12]:
@dataclass
class TimeResult:
    start_time: Optional[datetime]
    end_time: Optional[datetime]
    is_planned: bool
    raw_extract: Optional[str]
    method: str

    def to_dict(self) -> dict:
        d = asdict(self)
        d["start_time"] = self.start_time.isoformat() if self.start_time else None
        d["end_time"] = self.end_time.isoformat() if self.end_time else None
        return d

In [ ]:
# Parser

def _kyiv_now(base_dt: datetime) -> datetime:
    if base_dt.tzinfo is None:
        return KYIV_TZ.localize(base_dt)
    return base_dt.astimezone(KYIV_TZ)


def _make_dt(day: int, month: int, year: Optional[int],
             hour: int, minute: int, base_dt: datetime) -> datetime:

    base = _kyiv_now(base_dt)
    y = year if year else base.year

    try:
        dt = KYIV_TZ.localize(datetime(y, month, day, hour, minute))
    except ValueError:
        return None
    
    # If no year and the date is more than 30 days in the past, assume it's next year
    if not year and (base - dt).days > 30:
        try:
            dt = KYIV_TZ.localize(datetime(y + 1, month, day, hour, minute))
        except ValueError:
            pass

    return dt

def _parse_time(s: str) -> Tuple[int, int]:
    h, m = s.split(":")
    return int(h), int(m)

def _parse_dot_date(s: str, base_dt: datetime, hour=0, minute=0) -> Optional[datetime]:
    parts = s.strip().split(".")
    if len(parts) < 2:
        return None
    try:
        day = int(parts[0])
        month = int(parts[1])
        year = int(parts[2]) if len(parts) >= 3 and parts[2] else None
    except ValueError:
        return None
    return _make_dt(day, month, year, hour, minute, base_dt)

def _parse_word_date(day_str: str, month_str: str, year_str: Optional[str],
                     base_dt: datetime, hour=0, minute=0) -> Optional[datetime]:
    month = UK_MONTH_MAP.get(month_str.lower())
    if not month:
        return None
    try:
        day = int(day_str)
        year = int(year_str) if year_str else None
    except (ValueError, TypeError):
        return None
    return _make_dt(day, month, year, hour, minute, base_dt)


In [ ]:
#Patterns for ranges 

RANGE_PATTERNS = [
    # "з 18 березня 2025 року, 09:30 до 20 березня 18:00"
    # "з 18 березня, 09:30 до 20 березня 18:00"
    (
        r"з\s+(?P<sd>\d{1,2})\s+(?P<sm>" + UK_MONTHS + r")(?:\s+(?P<sy>\d{4})(?:\s+року)?)?\s*,?\s*(?P<st>\d{1,2}:\d{2})\s+до\s+(?P<ed>\d{1,2})\s+(?P<em>" + UK_MONTHS + r")(?:\s+(?P<ey>\d{4})(?:\s+року)?)?\s*,?\s*(?P<et>\d{1,2}:\d{2})",
        "word_date_time_range"
    ),
    # "з 9.02 09:00 до 15.02 18:00"
    (
        r"з\s+(?P<sd>\d{1,2}\.\d{1,2}(?:\.\d{4})?)\s+(?P<st>\d{1,2}:\d{2})\s+до\s+(?P<ed>\d{1,2}\.\d{1,2}(?:\.\d{4})?)\s+(?P<et>\d{1,2}:\d{2})",
        "dot_date_time_range"
    ),
    # "з 09:00 до 18:00" / "від 10:00 до 22:00"
    (r"(?:з|від)\s+(?P<start>\d{1,2}:\d{2})\s+до\s+(?P<end>\d{1,2}:\d{2})", "time_range"),
    # "09:00–18:00"
    (r"(?P<start>\d{1,2}:\d{2})\s*[-–—]\s*(?P<end>\d{1,2}:\d{2})", "time_range"),
    # "з 9.02 до 15.02" 
    (
        r"з\s+(?P<sd>\d{1,2}\.\d{1,2}(?:\.\d{4})?)\s*(?:до|по|–|—)\s+(?P<ed>\d{1,2}\.\d{1,2}(?:\.\d{4})?)",
        "dot_date_range"
    ),
    # "з 4 березня по 12 березня"
    (
        r"з\s+(?P<sd>\d{1,2})\s+(?P<sm>" + UK_MONTHS + r")(?:\s+(?P<sy>\d{4}))?\s*(?:до|по|–|—)\s+(?P<ed>\d{1,2})\s+(?P<em>" + UK_MONTHS + r")(?:\s+(?P<ey>\d{4}))?",
        "word_date_range"
    ),
    # "з 18 квітня 09:30 до 20-го квітня" (кінець без часу)
    (
        r"з\s+(?P<sd>\d{1,2})\s+(?P<sm>" + UK_MONTHS + r")(?:\s+(?P<sy>\d{4}))?\s*,?\s*(?P<st>\d{1,2}:\d{2})\s+до\s+(?P<ed>\d{1,2})(?:-го|-ї|-ого)?\s+(?P<em>" + UK_MONTHS + r")(?:\s+(?P<ey>\d{4}))?",
        "word_date_time_range_no_end_time"
    ),
    # "з 5 по 12 число"
    (r"з\s+(?P<sd>\d{1,2})\s+по\s+(?P<ed>\d{1,2})\s+число", "day_range"),
]

# single patterns
# only if range patterns didn't match. 
SINGLE_PATTERNS = [
    # "до 20:00" 
    (r"до\s+(?P<end_time>\d{1,2}:\d{2})", "end_time"),
    # "о 10:00" / "з 10:00"
    (r"(?:о|з)\s+(?P<time>\d{1,2}:\d{2})", "start_time"),
    # "18 березня 2025"
    (r"(?P<d>\d{1,2})\s+(?P<m>" + UK_MONTHS + r")(?:\s+(?P<y>\d{4}))?", "word_date"),
    # "18.03.2025"
    (r"(?P<dot_date>\d{1,2}\.\d{1,2}\.\d{4})", "dot_date"),
]



In [ ]:
def _handle_range(m: re.Match, handler: str, base_dt: datetime) -> Optional[Tuple[datetime, datetime]]:
    gd = m.groupdict()

    if handler == "time_range":
        sh, sm = _parse_time(gd["start"])
        eh, em = _parse_time(gd["end"])
        base = _kyiv_now(base_dt)
        start = KYIV_TZ.localize(datetime(base.year, base.month, base.day, sh, sm))
        end   = KYIV_TZ.localize(datetime(base.year, base.month, base.day, eh, em))
        if end < start:
            end += timedelta(days=1)
        return start, end

    if handler == "dot_date_range":
        start = _parse_dot_date(gd["sd"], base_dt)
        end   = _parse_dot_date(gd["ed"], base_dt)
        return (start, end) if start and end else None

    if handler == "dot_date_time_range":
        sh, sm = _parse_time(gd["st"])
        eh, em = _parse_time(gd["et"])
        start = _parse_dot_date(gd["sd"], base_dt, sh, sm)
        end   = _parse_dot_date(gd["ed"], base_dt, eh, em)
        return (start, end) if start and end else None

    if handler == "word_date_range":
        start = _parse_word_date(gd["sd"], gd["sm"], gd.get("sy"), base_dt)
        end   = _parse_word_date(gd["ed"], gd["em"], gd.get("ey"), base_dt)
        return (start, end) if start and end else None

    if handler == "word_date_time_range":
        sh, sm = _parse_time(gd["st"])
        eh, em = _parse_time(gd["et"])
        start = _parse_word_date(gd["sd"], gd["sm"], gd.get("sy"), base_dt, sh, sm)
        end   = _parse_word_date(gd["ed"], gd["em"], gd.get("ey"), base_dt, eh, em)
        return (start, end) if start and end else None

    if handler == "word_date_time_range_no_end_time":
        sh, sm = _parse_time(gd["st"])
        start = _parse_word_date(gd["sd"], gd["sm"], gd.get("sy"), base_dt, sh, sm)
        end   = _parse_word_date(gd["ed"], gd["em"], gd.get("ey"), base_dt)
        return (start, end) if start and end else None

    if handler == "day_range":
        base = _kyiv_now(base_dt)
        start = KYIV_TZ.localize(datetime(base.year, base.month, int(gd["sd"])))
        end   = KYIV_TZ.localize(datetime(base.year, base.month, int(gd["ed"])))
        return start, end

    return None


def _try_range(text: str, base_dt: datetime):
    for pattern, handler in RANGE_PATTERNS:
        m = re.search(pattern, text, re.IGNORECASE)
        if not m:
            continue
        result = _handle_range(m, handler, base_dt)
        if result:
            start, end = result
            if end < start:
                end += timedelta(days=1)
            return start, end, m.group(0)
    return None


def _try_single(text: str, base_dt: datetime):
    for pattern, handler in SINGLE_PATTERNS:
        m = re.search(pattern, text, re.IGNORECASE)
        if not m:
            continue
        gd = m.groupdict()
        base = _kyiv_now(base_dt)

        if handler == "end_time":
            h, mn = _parse_time(gd["end_time"])
            dt = KYIV_TZ.localize(datetime(base.year, base.month, base.day, h, mn))
            return None, dt, m.group(0)

        if handler == "start_time":
            h, mn = _parse_time(gd["time"])
            dt = KYIV_TZ.localize(datetime(base.year, base.month, base.day, h, mn))
            return dt, None, m.group(0)

        if handler == "word_date":
            dt = _parse_word_date(gd["d"], gd["m"], gd.get("y"), base_dt)
            if dt:
                return dt, None, m.group(0)

        if handler == "dot_date":
            dt = _parse_dot_date(gd["dot_date"], base_dt)
            if dt:
                return dt, None, m.group(0)

    return None

def _try_relative(text: str, base_dt: datetime) -> Optional[datetime]:
    base = _kyiv_now(base_dt)
    t = text.lower()
    if "вчора" in t:
        return KYIV_TZ.localize(datetime(base.year, base.month, base.day)) - timedelta(days=1)
    if "завтра" in t:
        return KYIV_TZ.localize(datetime(base.year, base.month, base.day)) + timedelta(days=1)
    if any(h in t for h in ["сьогодні", "зранку", "ввечері", "до вечора", "до кінця дня"]):
        return KYIV_TZ.localize(datetime(base.year, base.month, base.day))
    return None

def _next_midnight(dt: datetime) -> datetime:
    next_day = (dt + timedelta(days=1)).date()
    return KYIV_TZ.localize(datetime(next_day.year, next_day.month, next_day.day, 0, 0))


In [ ]:
def extract_time(text: str, published_at: datetime) -> TimeResult:
    base_dt = _kyiv_now(published_at) if published_at else _kyiv_now(datetime.now())
    is_planned = bool(PLANNED_RE.search(text))

    range_res = _try_range(text, base_dt)
    if range_res:
        start, end, raw = range_res
        return TimeResult(start, end, is_planned, raw, "range")

    single_res = _try_single(text, base_dt)
    if single_res:
        start, end, raw = single_res
        if start and not end:
            end = start + timedelta(hours=24)
            # end = _next_midnight(start) 
            method = "start_only"
        elif end and not start:
            start = base_dt
            method = "end_only"
        else:
            method = "date_only"
        return TimeResult(start, end, is_planned, raw, method)

    rel_dt = _try_relative(text, base_dt)
    if rel_dt:
        return TimeResult(rel_dt, rel_dt + timedelta(hours=24), is_planned, None, "relative")

    return TimeResult(base_dt, base_dt + timedelta(hours=24), is_planned, None, "none")

    #if rel_dt:
    #    return TimeResult(rel_dt, _next_midnight(rel_dt), is_planned, None, "relative")
    #return TimeResult(base_dt, _next_midnight(rel_dt), is_planned, None, "none")


In [18]:
test_cases = [
    "Планові роботи з 09:00 до 18:00 на Харківській",
    "Відключення води з 9.02 до 15.02",
    "Ремонт з 4 березня по 12 березня",
    "Перекриття з 5 по 12 число",
    "З 18 березня 2025 року, 09:30 до 20 березня 18:00",
    "З 18 квітня 09:30 до 20-го квітня",
    "Аварійні роботи до 20:00",
    "Сьогодні зранку аварія на Борщагівці",
    "Буде проведено планове відключення з 10:00 до 14:00",
    "Зіткнення на Голосіївському, рух ускладнено",
]

for text in test_cases:
    base_dt = datetime.now()
    r = extract_time(text, base_dt)
    print(f"\nТекст:    {text[:80]}")
    print(f"Метод:    {r.method:<12}  Планова: {r.is_planned}")
    print(f"Start:    {r.start_time}")
    print(f"End:      {r.end_time}")
    if r.raw_extract:
        print(f"Знайдено: '{r.raw_extract}'")


Текст:    Планові роботи з 09:00 до 18:00 на Харківській
Метод:    range         Планова: True
Start:    2026-04-07 09:00:00+03:00
End:      2026-04-07 18:00:00+03:00
Знайдено: 'з 09:00 до 18:00'

Текст:    Відключення води з 9.02 до 15.02
Метод:    range         Планова: False
Start:    2027-02-09 00:00:00+02:00
End:      2027-02-15 00:00:00+02:00
Знайдено: 'з 9.02 до 15.02'

Текст:    Ремонт з 4 березня по 12 березня
Метод:    range         Планова: False
Start:    2027-03-04 00:00:00+02:00
End:      2026-03-13 00:00:00+02:00
Знайдено: 'з 4 березня по 12 березня'

Текст:    Перекриття з 5 по 12 число
Метод:    range         Планова: False
Start:    2026-04-05 00:00:00+03:00
End:      2026-04-12 00:00:00+03:00
Знайдено: 'з 5 по 12 число'

Текст:    З 18 березня 2025 року, 09:30 до 20 березня 18:00
Метод:    range         Планова: False
Start:    2025-03-18 09:30:00+02:00
End:      2026-03-20 18:00:00+02:00
Знайдено: 'З 18 березня 2025 року, 09:30 до 20 березня 18:00'

Текст:    З 18 